# AtaCNV – DV90 scATAC-seq CNV inference

**Kernel: R**

La count matrix viene costruita da uno script Python (`build_matrix.py`) chiamato tramite `system()` nella Cella 4.

**Input richiesti:**
- `/sharedFolder/Data/dv90_atac_fragments.tsv.gz`
- `/sharedFolder/Data/filtered_feature_bc_matrix/barcodes.tsv.gz`
- `build_matrix.py` nella stessa cartella di questo notebook

**Output:** `/sharedFolder/Results/AtaCNV/DV90/`

---
⚠️ **Esegui le celle in ordine. La Cella 4 impiega ~4 minuti.**

## Cella 1 – Percorsi e cartelle output

In [37]:
# ── Percorsi input ────────────────────────────────────────
FRAG_FILE     <- "/sharedFolder/Data/dv90_atac_fragments.tsv.gz"
BARCODES_FILE <- "/sharedFolder/Data/filtered_feature_bc_matrix/barcodes.tsv.gz"

# ── Percorsi output ───────────────────────────────────────
OUT_DIR       <- "/sharedFolder/Results/AtaCNV/DV90"
BIN_CSV       <- file.path(OUT_DIR, "bin_info_hg38.csv")
COUNT_RDS     <- file.path(OUT_DIR, "count.rds")
NORM_RDS      <- file.path(OUT_DIR, "normalize", "norm_re.rds")
CNV_RDS       <- file.path(OUT_DIR, "cnv", "CNV_re.rds")
CN_STATE_RDS  <- file.path(OUT_DIR, "CN_state.rds")

# Script Python (deve essere nella stessa cartella del notebook)
PY_SCRIPT <- file.path(getwd(), "build_matrix.py")

# ── Crea cartelle ─────────────────────────────────────────
dir.create(file.path(OUT_DIR, "normalize"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(OUT_DIR, "cnv"),       recursive = TRUE, showWarnings = FALSE)

# ── Verifica file input ───────────────────────────────────
cat("=== File input ===\n")
for (f in c(FRAG_FILE, BARCODES_FILE, PY_SCRIPT)) {
  status <- if (file.exists(f)) "OK" else "*** NON TROVATO ***"
  cat(sprintf("  %-55s %s\n", basename(f), status))
}

barcodes <- readLines(gzfile(BARCODES_FILE))
cat(sprintf("\nBarcodes filtrati: %d\n", length(barcodes)))

=== File input ===
  dv90_atac_fragments.tsv.gz                              OK
  barcodes.tsv.gz                                         OK
  build_matrix.py                                         OK

Barcodes filtrati: 3472


## Cella 2 – Esporta bin_info_hg38 (reference bins 1Mb per AtaCNV)

In [38]:
library(AtaCNV)

data("bin_info_hg38", package = "AtaCNV")
write.csv(bin_info_hg38, BIN_CSV, row.names = FALSE, quote = FALSE)

cat(sprintf("Bin totali: %d\n", nrow(bin_info_hg38)))
cat("Colonne:", paste(names(bin_info_hg38), collapse = ", "), "\n")
print(head(bin_info_hg38, 3))
cat(sprintf("\nSalvato: %s\n", BIN_CSV))

Bin totali: 3102
Colonne: chr, start, end, arm, map, gene, gc 
   chr start   end arm    map gene       gc
1 chr1 0e+00 1e+06   p 247118   37 0.383494
2 chr1 1e+06 2e+06   p 867666   60 0.572853
3 chr1 2e+06 3e+06   p 858952   16 0.558315

Salvato: /sharedFolder/Results/AtaCNV/DV90/bin_info_hg38.csv


## Cella 3 – Verifica dipendenze Python
Controlla che pandas, numpy e pyreadr siano installati.

In [39]:
cat("=== Verifica dipendenze Python ===\n")
ret <- system("python3 -c 'import pandas, numpy, pyreadr; print(\"OK: pandas\", pandas.__version__, \"| numpy\", numpy.__version__, \"| pyreadr\", pyreadr.__version__)'",
              intern = TRUE)
cat(ret, "\n")
# Se manca qualcosa: system("pip install pyreadr pandas numpy")

=== Verifica dipendenze Python ===
OK: pandas 2.3.3 | numpy 2.2.6 | pyreadr 0.5.6 


## Cella 4 – Costruisce la count matrix (~4 minuti)
Lancia `build_matrix.py` che:
- Legge il fragment file in chunk da 5M righe
- Filtra automaticamente cromosomi alternativi (usa solo i chr presenti in bin_info_hg38)
- Salva `count.rds` (celle × bin)

In [40]:
cmd <- paste("python3", PY_SCRIPT, FRAG_FILE, BARCODES_FILE, BIN_CSV, COUNT_RDS)
cat("Comando:", cmd, "\n\n")

ret <- system(cmd, intern = TRUE)
cat(paste(ret, collapse = "\n"), "\n")

if (file.exists(COUNT_RDS)) {
  cat(sprintf("\n✓ count.rds creato (%.0f MB)\n", file.size(COUNT_RDS)/1e6))
} else {
  stop("ERRORE: count.rds non trovato. Controlla l'output sopra.")
}

Comando: python3 /sharedFolder/Script/build_matrix.py /sharedFolder/Data/dv90_atac_fragments.tsv.gz /sharedFolder/Data/filtered_feature_bc_matrix/barcodes.tsv.gz /sharedFolder/Results/AtaCNV/DV90/bin_info_hg38.csv /sharedFolder/Results/AtaCNV/DV90/count.rds 

Caricamento bin_info_hg38...
Bin: 3102 | Cromosomi validi: 24
Barcodes: 3472
Parsing fragment file (chunked)...
  chunk 5: 25,000,000 letti, 22,035,971 mappati
  chunk 10: 50,000,000 letti, 44,036,279 mappati
  chunk 15: 75,000,000 letti, 65,941,692 mappati
  chunk 20: 100,000,000 letti, 88,450,678 mappati
  chunk 25: 125,000,000 letti, 110,396,967 mappati
  chunk 30: 150,000,000 letti, 132,047,649 mappati
  chunk 35: 175,000,000 letti, 153,521,924 mappati
  chunk 40: 200,000,000 letti, 175,112,843 mappati

TOTALE: 210,197,647 letti | 183,873,732 mappati (87.5%)
Celle non-zero: 3472 / 3472
Bin  non-zero:  2965 / 3102
Somma conteggi: 183,873,732
Salvato: /sharedFolder/Results/AtaCNV/DV90/count.rds 

✓ count.rds creato (43 MB)


## Cella 5 – Carica e verifica la count matrix

In [41]:
count_raw <- readRDS(COUNT_RDS)
barcodes  <- readLines(gzfile(BARCODES_FILE))
count     <- as.matrix(count_raw)
rownames(count) <- barcodes

cat("=== Count matrix ===\n")
cat(sprintf("Dimensioni:     %d celle x %d bin\n", nrow(count), ncol(count)))
cat(sprintf("Somma totale:   %s\n",  format(sum(count), big.mark = ",")))
cat(sprintf("Celle non-zero: %d / %d\n", sum(rowSums(count) > 0), nrow(count)))
cat(sprintf("Bin  non-zero:  %d / %d\n", sum(colSums(count) > 0), ncol(count)))
cat(sprintf("Range valori:   %d - %d\n", min(count), max(count)))
cat("Rownames (3):", head(rownames(count), 3), "\n")

=== Count matrix ===
Dimensioni:     3472 celle x 3102 bin
Somma totale:   183,873,732
Celle non-zero: 3472 / 3472
Bin  non-zero:  2965 / 3102
Range valori:   0 - 2486
Rownames (3): AAACAGCCACTGGCCA-1 AAACAGCCAGCAAGGC-1 AAACAGCCAGCAATAA-1 


## Cella 6 – Normalizzazione
`mode = "none"`: nessuna normal cell disponibile.
AtaCNV clusterizza automaticamente le cellule (K=5 cluster, hclust ward.D)
e seleziona come baseline il cluster con la correlazione più alta ai gene counts di hg38.

In [42]:
library(AtaCNV)

norm_re <- AtaCNV::normalize(
  count         = count,
  genome        = "hg38",
  mode          = "none",
  output_dir    = file.path(OUT_DIR, "normalize", ""),
  output_name   = "norm_re.rds",
  gc_correction = TRUE
)

cat("\n=== norm_re ===\n")
cat("Elementi:", paste(names(norm_re), collapse = ", "), "\n")
cat(sprintf("copy_ratio: %d x %d\n", nrow(norm_re$copy_ratio), ncol(norm_re$copy_ratio)))
cat(sprintf("norm_count: %d x %d\n", nrow(norm_re$norm_count), ncol(norm_re$norm_count)))
cat(sprintf("baseline:   %d valori\n", length(norm_re$baseline)))
cat(sprintf("Salvato: %s\n", NORM_RDS))

[1] "Step 1: filtering bins and cells..."
[1] " number of bin before filter: 3102"
[1] " number of bin after filter: 2799"
[1] " number of cells before filter: 3472"
[1] " number of cells after filter: 3472"
[1] "Step 2: smoothing read count signals..."
[1] "Step 3: calculating baseline of normal cells..."
[1] " Cell cluster not provided. Assigning cell clusters..."
[1] " Selecting normal cells using correlation with gene counts..."
[1] " cluster 5;441cells;corr=0.573494885705399"
[1] " cluster 2;1642cells;corr=0.546990645273544"
[1] " cluster 1;143cells;corr=0.556375546102612"
[1] " cluster 4;975cells;corr=0.548931451980794"
[1] " cluster 3;271cells;corr=0.59910034505132"
[1] " selected normal: 271 cells"
[1] " max correlation to gene counts: 0.59910034505132"
[1] " Baseline is calculated using median read count of selected normal cells"
[1] "Step 4: normalizing against baseline..."
[1] "99th percentile: 1.73562264546887"
[1] "Step 5: GC correction..."
[1] "Step 6: saving results..."


In [43]:
library(parallelDist)

# Stesso clustering che fa normalize() internamente
d       <- parallelDist::parallelDist(norm_re$norm_count)  # dist tra celle
hc_re   <- hclust(d, method = "ward.D")
cluster <- cutree(hc_re, k = 5)

cat("Distribuzione cluster:\n")
print(table(cluster))

# Il cluster normale è quello con correlazione più alta ai gene counts
# dall'output di normalize() era cluster 5 con 166 celle
# Costruisci label: "normal" per cluster 5, stringa numerica per gli altri
label <- ifelse(cluster == 5, "normal", as.character(cluster))
cat("\nLabel 'normal':", sum(label == "normal"), "cellule\n")
cat("Label tumorali:", sum(label != "normal"), "cellule\n")

Distribuzione cluster:
cluster
   1    2    3    4    5 
 403  837 1917  136  179 

Label 'normal': 179 cellule
Label tumorali: 3293 cellule


## Cella 7 – Segmentazione CNV con BICseq2
Segmenta ogni cellula lungo il genoma e calcola il copy ratio per segmento.

In [44]:
seg_re <- AtaCNV::calculate_CNV(
  norm_count  = norm_re$norm_count,
  baseline    = norm_re$baseline,
  output_dir  = file.path(OUT_DIR, "cnv", ""),
  output_name = "CNV_re.rds"
)

cat("\n=== seg_re ===\n")
cat("Elementi:", paste(names(seg_re), collapse = ", "), "\n")
cat(sprintf("Breakpoints totali: %d\n", length(seg_re$bkp)))
cat(sprintf("copy_ratio: %d x %d\n", nrow(seg_re$copy_ratio), ncol(seg_re$copy_ratio)))
cat("Primi 10 breakpoints:", head(seg_re$bkp, 10), "\n")
cat(sprintf("Salvato: %s\n", CNV_RDS))

[1] "Running BICseq2 for segmentation..."
[1] "chr1"
[1] "chr2"
[1] "chr3"
[1] "chr4"
[1] "chr5"
[1] "chr6"
[1] "chr7"
[1] "chr8"
[1] "chr9"
[1] "chr10"
[1] "chr11"
[1] "chr12"
[1] "chr13"
[1] "chr14"
[1] "chr15"
[1] "chr16"
[1] "chr17"
[1] "chr18"
[1] "chr19"
[1] "chr20"
[1] "chr21"
[1] "chr22"
[1] "chrX"
[1] "chrY"
[1] "number of breakpoints: 50"
[1] "Saving results..."

=== seg_re ===
Elementi: copy_ratio, bkp, CNV_seg 
Breakpoints totali: 50
copy_ratio: 3472 x 2799
Primi 10 breakpoints: 0 52 112 177 219 321 362 451 509 550 
Salvato: /sharedFolder/Results/AtaCNV/DV90/cnv/CNV_re.rds


## Cella 8 – Stima stati CN discreti
Inferisce gli stati di copy number (0, 1, 2, 3, 4+) per ogni cellula e segmento.

In [45]:
library(parallelDist)
library(AtaCNV)
data("bin_info_hg38", package = "AtaCNV")

d       <- parallelDist::parallelDist(norm_re$norm_count)
hc_re   <- hclust(d, method = "ward.D")
cluster <- cutree(hc_re, k = 5)

gene_ref <- bin_info_hg38$gene
corr <- sapply(1:5, function(k) {
  f   <- cluster == k
  med <- if (sum(f) == 1) norm_re$norm_count[f, ] else colMeans(norm_re$norm_count[f, ])
  cor(med, gene_ref[1:length(med)], method = "pearson")
})
normal_cluster <- which.max(corr)
cat(sprintf("Cluster riferimento: %d (%d celle, corr=%.3f)\n",
            normal_cluster, sum(cluster == normal_cluster), max(corr)))

label <- ifelse(cluster == normal_cluster, "normal", as.character(cluster))

CN_state <- AtaCNV::estimate_cnv_state_cluster(
  count      = count,
  genome     = "hg38",
  copy_ratio = norm_re$copy_ratio,
  bkp        = seg_re$bkp,
  label      = label
)
saveRDS(CN_state, file.path(OUT_DIR, "CN_state.rds"))
cat("Salvato CN_state.rds\n")

Cluster riferimento: 4 (136 celle, corr=0.038)
[1] "Estimating copy ratio distribution..."


simATAC is:

...estimating library size...

...estimating non-zero cell proportion...

...estimating bin mean...

simATAC is:

...updating parameters...

...setting up SingleCellExperiment object...

Your data has different number of bins compared to the provided genome positions. Please give a file of bin information consistent with your input data with three columns and header of "chr start end" as the bin.coordinate.file parameter. If you don't give a file containing the information of bins, simATAC considers the bin.coordinate.file parameter as "None" and names the bins {Bin1 to BinX} with X number of bins. In this case, you wont be able to get the coordinate information of bins. Please make sure the "species" parameter of the simATACCount object is set correctly.

...simulating library size...

...simulating non-zero cell proportion...

...simulating bin mean...

...generating final counts...

...Done...



[1]  500 3102


simATAC is:

...estimating library size...

...estimating non-zero cell proportion...

...estimating bin mean...

simATAC is:

...updating parameters...

...setting up SingleCellExperiment object...

Your data has different number of bins compared to the provided genome positions. Please give a file of bin information consistent with your input data with three columns and header of "chr start end" as the bin.coordinate.file parameter. If you don't give a file containing the information of bins, simATAC considers the bin.coordinate.file parameter as "None" and names the bins {Bin1 to BinX} with X number of bins. In this case, you wont be able to get the coordinate information of bins. Please make sure the "species" parameter of the simATACCount object is set correctly.

...simulating library size...

...simulating non-zero cell proportion...

...simulating bin mean...

...generating final counts...

...Done...



[1]  500 3102


simATAC is:

...estimating library size...

...estimating non-zero cell proportion...

...estimating bin mean...

simATAC is:

...updating parameters...

...setting up SingleCellExperiment object...

Your data has different number of bins compared to the provided genome positions. Please give a file of bin information consistent with your input data with three columns and header of "chr start end" as the bin.coordinate.file parameter. If you don't give a file containing the information of bins, simATAC considers the bin.coordinate.file parameter as "None" and names the bins {Bin1 to BinX} with X number of bins. In this case, you wont be able to get the coordinate information of bins. Please make sure the "species" parameter of the simATACCount object is set correctly.

...simulating library size...

...simulating non-zero cell proportion...

...simulating bin mean...

...generating final counts...

...Done...



[1]  500 3102
[1] "Step 1: filtering bins and cells..."
[1] " number of bin before filter: 3102"
[1] " number of bin after filter: 2799"
[1] " number of cells before filter: 500"
[1] " number of cells after filter: 500"
[1] "Step 2: smoothing read count signals..."
[1] "Step 3: calculating baseline of normal cells..."
[1] " Baseline is calculated using median read count of normal cells"
[1] "Step 4: normalizing against baseline..."
[1] "99th percentile: 1.49563581117966"
[1] "Step 5: GC correction..."
[1] "Step 6: saving results..."
[1] "Assigning initial copy number state..."
[1] "Number of diploid region: 1969"
[1] "Adjusting result..."
[1] "Number of diploid region: 1798"
[1] "Adjusting result..."
Salvato CN_state.rds


## Cella 9 – Plot heatmap CNV

In [46]:
options(repr.plot.width = 16, repr.plot.height = 8)

AtaCNV::plot_heatmap(
  copy_ratio  = seg_re$copy_ratio,
  output_dir  = file.path(OUT_DIR, "cnv", ""),
  output_name = "DV90_CNV_heatmap.png"
)

[1] "Step 1: clustering..."
[1] "Step 2: plotting..."


'magick' package is suggested to install to give better rasterization.

Set `ht_opt$message = FALSE` to turn off this message.



## Cella 10 – Riepilogo file output

In [47]:
cat("=== Output in:", OUT_DIR, "===\n\n")
files <- list.files(OUT_DIR, recursive = TRUE, full.names = TRUE)
files <- files[!file.info(files)$isdir]
for (f in files) {
  cat(sprintf("  %-45s %6.1f MB\n",
              sub(OUT_DIR, "", f),
              file.size(f) / 1e6))
}

=== Output in: /sharedFolder/Results/AtaCNV/DV90 ===

  /bin_info_hg38.csv                               0.1 MB
  /CN_state.rds                                   73.3 MB
  /cnv/CNV_re.rds                                  2.3 MB
  /cnv/DV90_CNV_heatmap.png                        0.8 MB
  /count.rds                                      43.3 MB
  /normalize/norm_re.rds                         215.3 MB


In [48]:
# In R – esporta copy_ratio come CSV leggibile da Python
seg_re <- readRDS("/sharedFolder/Results/AtaCNV/DV90/cnv/CNV_re.rds")
cat("Elementi:", names(seg_re), "\n")
cat("copy_ratio dim:", dim(seg_re$copy_ratio), "\n")

write.csv(seg_re$copy_ratio, 
          "/sharedFolder/Results/AtaCNV/DV90/cnv/copy_ratio.csv",
          row.names = TRUE)
cat("Salvato copy_ratio.csv\n")

Elementi: copy_ratio bkp CNV_seg 
copy_ratio dim: 3472 2799 
Salvato copy_ratio.csv
